In [7]:
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

data = pd.read_csv("../Data/Raw/train.csv")

def bmi_category(bmi):
    if bmi < 16:
        return "Severe_Thinness"
    elif bmi < 18.5:
        return "Underweight"
    elif bmi < 22:
        return "Normal_Low"
    elif bmi < 25:
        return "Normal_High"
    elif bmi < 27.5:
        return "Overweight_Low"
    elif bmi < 30:
        return "Overweight_High"
    elif bmi < 35:
        return "Obese"
    else:
        return "Severe_Obese"

def hear_rate_category(hr):
    if hr < 60:
        return "Low"
    elif hr <= 120: 
        return "Normal"
    else:
        return "High"

def activity_category(steps):
    if steps < 5000:
        return "Low"
    elif steps < 10000:
        return "moderate"
    else:
        return "High"
    
def exercise_category(exercise):
    if exercise == 0:
        return "None"
    elif exercise < 30:
        return "Short"
    elif exercise < 150:
        return "Normal"
    else:
        return "High"

def water_intake(water):
    if water < 1.5:
        return "Low"
    elif water < 3:
        return "Normal"
    else:
        return "High"

data["bmi_category"] = data["bmi"].apply(bmi_category)
data["heart_rate_category"] = data["heart_rate"].apply(hear_rate_category)
data["activity_category"] = data["step_count"].apply(activity_category)
data["exercise_category"] = data["exercise_duration"].apply(exercise_category)
data["water_category"] = data["water_intake"].apply(water_intake)
numerical_data = ["sleep_duration", "heart_rate", "bmi", "step_count", "exercise_duration"
                  ,"water_intake", "calorie_expenditure"]

categorical_data = ["diet_type", "stress_level", "sleep_quality",
                     "physical_activity_level", "smoking_alcohol",
                       "gender", "bmi_category", "heart_rate_category", "activity_category", "water_category"]

y = data["health_condition"]
X = data[numerical_data + categorical_data]

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer(
    transformers = [(
        "cat",
        OneHotEncoder(handle_unknown="ignore"),
        categorical_data
    )],
    remainder="passthrough"
)

model = Pipeline([
    ("prepocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42, n_jobs=-1))
])

model.fit(X_train, y_train)

scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")

print("CV Accuracy:", scores)
print("Mean", scores.mean())
print("Std:", scores.std())

CV Accuracy: [0.96554073 0.96624353 0.96632323 0.9658303  0.96581581]
Mean 0.9659507189585439
Std: 0.0002916519383893751
